# Feature Engineering — Documentation

> **The production source of truth is now [`betting/features.py`](features.py)**, not this
> notebook. The feature logic was extracted to an importable, diffable, testable module so
> notebook-storage quirks can no longer affect production. The hermetic synthetic-data tests
> live in [`betting/test_features.py`](test_features.py) and run in CI.

This notebook is kept as **design documentation** — the markdown below explains each of the
85-feature groups (1–10) and the pipeline. The import cell makes the functions available for
interactive exploration. Edit feature logic in `features.py`, not here.


In [ ]:
# Documentation notebook — the production code lives in betting/features.py.
# This cell just imports it so the markdown below stays runnable for exploration.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() if (Path.cwd() / 'features.py').exists() else Path.cwd() / 'betting'))
from features import *          # build_features, build_numeric_features, constants, helpers
print('Imported betting/features.py. Run tests with:  pytest betting/test_features.py')

## Constants

* **`TEAM_MAP`** — folds historical / alternate NFL team abbreviations into the
  modern set so AllPro CSV and PBP tables join cleanly with the schedule.
* **`FEATURE_COLS_85`** — the canonical ordered list of all 85 engineered
  features, exactly matching the column names the production pkls were trained on.
  The trailing space in `"allpro_diff_home_def_away_off_3_years "` is **intentional**
  and must not be stripped without retraining.
* **`PROD_FEATURES_35`** — top-35 subset by combined XGB gain + Ridge |coef| +
  LGB gain, picked by the 2026-05-20 ablation study. Engineering still produces
  all 85; only these 35 are passed to model training.


## Helper — `norm_name`

Normalises a player name (strips Jr./Sr./roman-numeral suffixes, accents,
punctuation; lowercases; collapses whitespace). Used to join the AllPro CSV
(`P. Mahomes`) with the injury report (`Patrick Mahomes Jr.`).

**Inputs:** any string, including non-strings (returns `""`).
**Output:** normalised lowercase string.
**Test:** asserts suffix-stripping, accent-removal, punctuation-removal,
non-string safety.


## Helper — `canonicalize_ngs_team`

NFL Next Gen Stats labels every season with the **modern** team abbreviation
(`LAR`, `LV`, `LAC`) even for years when the team played under a different
abbreviation. This helper maps NGS abbrevs back to the schedule's per-season
abbreviation so the team merge in `_build_passer_rating` succeeds.

**Inputs:** `team_abbr` (str), `season` (int).
**Output:** the schedule-convention abbrev for that team in that season.
**Test:** asserts the three known relocations + a no-op passthrough.


## Group 1 — `_build_schedule_context`

Adds `is_playoff` and `is_final_week` boolean flags to the upcoming-week rows.
Warns if the target week contains playoff games, since the production models
were trained on REG-season only.

**Inputs:** `upcoming` (target-week games), `full_schedule`, `target_week`.
**Outputs:** `upcoming` with `is_playoff`, `is_final_week` columns.
**Test:** asserts both flag columns are present and bool-typed; final-week
detection is correct (the synth target_week=5 with REG max_week=4 → not final).


## Group 2 — `_build_rolling_pbp`

Adds 5-game rolling EPA, yards/play, and play-count for each team, both
offensive (`home/away_rolling_avg_epa`, ...) and defensive
(`home/away_rolling_allowed_avg_epa`, ...). Then computes 6 cross-matchup
diffs (off-vs-opponent-def + def-vs-opponent-off, for each of EPA / yards /
play count).

**Inputs:** `upcoming`, `pbp_s` (filtered PBP), `wk_lookup` (game_id → week/season).
**Outputs:** `upcoming` with 12 rolling cols + 6 diff cols.
**Test:** asserts all 18 expected columns exist and the diff columns equal
`home_X - away_Y` (i.e. the diff math wasn't reversed).


## Groups 3 & 5 — `_build_sos_and_performance`

Combined because Group 5 builds on the long-format team-game DataFrame Group 3
constructs. Adds:

* **SOS** — `home/away_recent_sos_opponent_avg` (rolling-3 of opponent win%),
  `home/away_season_sos_opponent_avg` (expanding mean), `sos_diff`, `season_sos_diff`
* **Rolling win%** — `home/away_rolling_win_pct` (5-game window)
* **Scoring** — `home/away_rolling_scored`, `home/away_rolling_allowed`, `scoring_diff`, `scoring_diff_reverse`
* **Cover rate** — `home/away_rolling_cover_rate`, `cover_rate_diff`
* **League margin** — `league_rolling_avg_abs_margin_by_week`

**Test:** asserts the headline 6 columns exist and are non-NaN for the synthetic
input (KC won every game → home_rolling_win_pct == 1.0).


## Group 4 — `_build_allpro`

Weighted 3-year AllPro roster quality, split by offense and defense. The
weighting is 4 / 2 / 1 for the most-recent / 2-years-ago / 3-years-ago AllPro
nominations. Plus prev-year counts (overall / offense / defense). Then computes
6 diff columns between home and away.

**Inputs:** `upcoming`, `allpro_df`, `target_season`.
**Outputs:** `upcoming` with 12 weighted/count columns + 6 diff columns.
**Test:** asserts all 18 columns present and the trailing-space column name is
preserved exactly.


## Group 6 — `_build_situational_pbp`

5-game rolling sacks, turnovers, and third-down conversion rate per team, then
6 diff columns (forward and reverse) for sack / turnover / 3rd-down.

**Inputs:** `upcoming`, `pbp_s`, `wk_lookup`.
**Outputs:** 6 diff columns plus the underlying `home/away_rolling_*` columns.
**Test:** asserts diff columns exist and the diff_reverse equals -diff.


## Group 7 — `_build_qb_switch`

Flags whether each team's QB this week is different from the QB they ended the
prior game with (`home/away_qb_switch`) and a parallel `is_home/away_qb_new`
flag (currently the same — kept distinct so the two pairs could diverge in a
future retrain).

**Inputs:** `upcoming`, `history`, optional `coach_hist_df` (full schedule with
prior-season results), `target_season`.
**Output:** 4 boolean columns.
**Test:** asserts both flag pairs are False when the synth QB never changes.


## Group 8 — `_build_passer_rating`

Prior-season passer rating + completion-% above expectation + avg time-to-throw,
sourced from NFL Next Gen Stats (2016+) with a manual-passer-rating fallback
for pre-2016 seasons (or any year where the NGS load fails). NGS abbrevs go
through `canonicalize_ngs_team` before the merge.

**Inputs:** `upcoming`, `pbp_rp`, `target_season`.
**Outputs:** 9 columns — `home/away/diff_pr_prev_year` + `..._cpae_prev_year` +
`..._time_to_throw_prev_year`.
**Test:** asserts all 9 columns exist and are non-NaN after median imputation.


## Group 9 — `_build_injuries`

Counts injured players (Out + Doubtful) per team for the target week, and the
AllPro-weighted injury impact (active AllPro weight = team's 3-year weight
minus the weight tied up in injured AllPros). Falls back to zero-fill on any
nflreadpy failure.

**Depends on Group 4** — reads `home/away_allpro_last_3_years_weighted` and
`home/away_allpro_prev_year` from `upcoming`.

**Inputs:** `upcoming` (with Group 4 cols), `allpro_df`, `target_season`, `target_week`.
**Outputs:** 5 columns — `home/away_injured_count`, `diff_injured_count`,
`diff_active_allpro_weighted`, `diff_active_allpro_prev_year`.
**Test:** asserts the 5 columns exist (values will be 0 in offline synth since
nflreadpy.load_injuries fails without network).


## Group 10 — `_build_coach_win_pct`

Career win% prior to this game + rolling 3-season win% for the home and away
head coach. Uses cumulative wins / games across the full coach history (1999+).

**Inputs:** `upcoming`, optional `coach_hist_df` (1999+ schedule with results),
`target_season`, `target_week`.
**Outputs:** 4 columns — `home/away_coach_win_pct_prior` + `..._roll3`.
**Test:** Andy Reid won every synth game → his roll3 should be 1.0.


## Main pipeline — `build_features`

Top-level entry point that calls each per-group helper in order. Group 9
(injuries) depends on Group 4 (AllPro) columns, so they must run in that
sequence — this is enforced by the call order below, not by any assertion in
the helpers themselves.

**Inputs:**
- `target_week`, `target_season` — the game to predict.
- `full_schedule` — DataFrame of schedules ≥ `target_season - 1`.
- `pbp_rp` — DataFrame of run/pass PBP for the current + prior season.
- `allpro_df` — AllPro CSV DataFrame, team-mapped via `TEAM_MAP`.
- `week_margin_lkp` — optional pd.Series indexed by week; if None, derived
  from current-season history.
- `coach_hist_df` — optional 1999+ schedule with results; if None, Group 10
  fetches live via `nflreadpy`.
- `required_features` — iterable of features that must be present at the end.
  Missing features are zero-filled with a warning. Defaults to `FEATURE_COLS_85`.

**Output:** the `upcoming` DataFrame with all 85 feature columns appended,
NaNs imputed with column-median. Returns `None` if no games for that week.

**Test:** synthetic-data integration test — asserts the result has all 85
expected features and none of the headline columns are NaN.


## `build_numeric_features` — categorical encode + numeric matrix

Used by the Ensemble and LightGBM voters. Takes the `upcoming` DataFrame,
ordinally encodes `roof` and `surface` against an already-fit `OrdinalEncoder`,
then assembles a float32 numeric matrix in the order specified by
`feature_cols`. Unknown / NaN categories fall back to the encoder's first known
category so the matrix stays dense.

**Inputs:** `upcoming_df`, `feature_cols` (ordered list), `enc` (fit OrdinalEncoder).
**Output:** float32 numpy array of shape `(len(upcoming_df), len(feature_cols))`.
**Test:** known-categories produces correct shape/dtype; unknown and NaN
inputs don't raise.


## Summary

Everything above is documentation for **`betting/features.py`**. Consumers (`predict_betting.ipynb`, `predict_totals.ipynb`, `model_comparison.ipynb`, `betting/experiments/*.py`) import that module directly — nothing loads this notebook. Run the real tests with `pytest betting/test_features.py`. If you reached this cell with no failed
assertions, the feature engineering is sound on synthetic data — production
correctness is then validated against live nflreadpy data in the consumer
notebook's own test cells.

**Don't change a feature name without retraining the production pkls.** The
trailing space in `allpro_diff_home_def_away_off_3_years ` is intentional and
matches the column the model was trained on.
